# 🔬 RAG-Fusion — Reciprocal Rank Fusion

RAG-Fusion starts exactly where **Multi-Query** ends: rewrite the question several ways, retrieve
for each. The difference is what happens next.

Multi-Query throws the rankings away — it takes a flat union, so a chunk ranked #1 by all five
variations is treated identically to one that scraped in at #4 of a single variation. **RAG-Fusion
keeps the rankings** and fuses them with *Reciprocal Rank Fusion* (RRF), so agreement across
queries becomes evidence of relevance.

## Learning Objectives
1. **The information Multi-Query discards** — why a flat union wastes the ranking signal
2. **The RRF formula** — `1 / (rank + k)`, and what the constant `k` actually controls
3. **Implementing fusion** — accumulate scores across result lists and re-rank
4. **Seeing the scores** — inspect which chunks several queries agreed on, and by how much
5. **Fusion vs union** — compare RAG-Fusion's ordering against a plain Multi-Query merge

## Prerequisites
- A `.env` at the repo root with `OPENAI_API_KEY` and `EXPERIENTIALLABS_API_KEY`
- Source documents in `04_Retrieval_and_RAG/shared_data/`
- **`a. Multi_Query.ipynb`** — RAG-Fusion is the same fan-out with a smarter merge

---
## 🧠 Part 1: What a Flat Union Throws Away

Multi-Query produces one ranked list per variation and then flattens them into a set. That merge
loses two useful signals:

| Signal | Meaning | Lost by a union? |
|---|---|---|
| **Rank within a list** | This chunk was the *best* match for that query | ✅ Lost |
| **Agreement across lists** | Several independent phrasings all found it | ✅ Lost |

A chunk that every variation ranked first is almost certainly relevant. A chunk that appeared once,
at position four, probably is not. After a flat union they are indistinguishable.

### Reciprocal Rank Fusion

RRF scores each document by summing a decreasing function of its rank in every list it appears in:

$$\text{score}(d) = \sum_{q \in \text{queries}} \frac{1}{\text{rank}_q(d) + k}$$

Two properties make this work well:

- **Rank, not score.** Similarity scores from different queries are not comparable — rank positions
  are. RRF never touches the raw distances.
- **Agreement compounds.** A document found by three queries adds three terms, so consensus
  naturally outranks a single strong hit.

The constant `k` (conventionally 60) flattens the curve. With `k = 60`, rank 0 scores `1/60 ≈
0.0167` and rank 3 scores `1/63 ≈ 0.0159` — close together, so **being found by many queries
matters more than placing first in any one of them**. A small `k` inverts that priority.

> **Key Insight**: RRF is a voting scheme. Each query casts rank-weighted votes, and documents that
> many queries vote for rise to the top.

---
## ⚙️ Part 2: Environment Setup

### 2.1 Imports

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
import os
import warnings

from pydantic import BaseModel, Field

# LangChain core
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Integrations
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Project helper (LLM factory)
from helpers import get_experientiallabs_llm

from dotenv import load_dotenv
from operator import itemgetter

warnings.filterwarnings("ignore")

print("✅ Imports loaded successfully!")

### 2.2 Credentials and LangSmith Tracing

> **Note**: use `LANGSMITH_PROJECT`, not the legacy `LANGCHAIN_PROJECT`. The SDK checks the
> `LANGSMITH_` prefix **first**, so a `LANGSMITH_PROJECT` already set in `.env` would silently win
> and these traces would land in that project instead of this one.

In [ ]:
# ============================================================================
# CONFIGURATION: Credentials and tracing
# ============================================================================
load_dotenv()

os.environ["LANGSMITH_PROJECT"] = "RAG-Fusion"

print(f"✅ OpenAI key present:  {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"✅ LangSmith tracing:   {os.getenv('LANGSMITH_TRACING')}")
print(f"✅ LangSmith project:   {os.environ['LANGSMITH_PROJECT']}")

### 2.3 Initialize the Models

In [ ]:
# ============================================================================
# MODEL INITIALIZATION: LLM (writes variations) + embeddings (search)
# ============================================================================
llm = get_experientiallabs_llm()

# Pinned explicitly: a bare OpenAIEmbeddings() still defaults to legacy ada-002.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"🤖 LLM:        {llm.model_name}")
print(f"🔢 Embeddings: {embeddings.model}")

---
## 📚 Part 3: Build the Knowledge Base

Identical to the Multi-Query notebook — same documents, same chunking — so the two techniques can
be compared on equal footing.

In [ ]:
# ============================================================================
# KNOWLEDGE BASE: Load, split, and index
# ============================================================================
loaders = [
    TextLoader("../../shared_data/blog.langchain.dev_announcing-langsmith_.txt", encoding="utf-8"),
    TextLoader("../../shared_data/blog.langchain.dev_automating-web-research_.txt", encoding="utf-8"),
]

docs = []
for loader in loaders:
    docs.extend(loader.load())

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=400,    # tokens, not characters
    chunk_overlap=60,  # keeps ideas intact across boundaries
)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()

print(f"📄 Loaded {len(docs)} documents → {len(splits)} chunks → indexed in Chroma")

---
## ✍️ Part 4: Generate Query Variations

RAG-Fusion typically uses **more** variations than Multi-Query (five rather than three) — fusion
needs several rankings before agreement carries any information.

### 4.1 Why Structured Output, Not `split("\n")`

The original version of this notebook parsed the LLM's reply with `(lambda x: x.split("\n"))`,
which assumes the model returns exactly N bare lines. Models routinely add headers, numbering, or a
preamble — and each of those lines then becomes a "query" sent to the retriever.

For RAG-Fusion the damage is worse than for Multi-Query: junk entries produce junk ranked lists,
and those lists **still vote** in the fusion step. Bad parsing does not merely add noise, it
corrupts the ranking.

`with_structured_output()` binds a Pydantic schema, so the provider returns validated JSON and you
get a guaranteed `list[str]`.

In [ ]:
# ============================================================================
# QUERY GENERATION: Schema-validated variations
# ============================================================================
class QueryVariations(BaseModel):
    """Alternative phrasings of the user's question."""

    queries: list[str] = Field(
        description="Five alternative phrasings, each targeting a different facet of the intent"
    )


template = """You are helping a vector search engine find relevant documents.

The user asked: "{question}"

Write five alternative phrasings of this question. Each should target a different
facet of the user's intent and use different vocabulary, so that documents are found
even when they do not share keywords with the original question."""

rag_fusion_prompt_template = ChatPromptTemplate.from_template(template)

generate_queries = (
    rag_fusion_prompt_template
    | llm.with_structured_output(QueryVariations)
    | (lambda x: x.queries)
)

print("✅ Query generation chain ready")

In [ ]:
# ============================================================================
# INSPECTION: What variations did the LLM produce?
# ============================================================================
question = "What is LangSmith, and why do we need it?"

variations = generate_queries.invoke({"question": question})

print(f"❓ Original: {question}\n")
print(f"📋 {len(variations)} generated variations:")
for i, q in enumerate(variations, 1):
    print(f"   {i}. {q}")

---
## 🗂️ Part 5: Retrieve a Ranked List per Variation

`retriever.map()` runs the retriever once per variation. Crucially, **each result list stays
ordered** — that ordering is the raw material RRF consumes.

In [ ]:
# ============================================================================
# FAN-OUT RETRIEVAL: One RANKED list per variation
# ============================================================================
per_query_docs = retriever.map().invoke(variations)

print(f"🔍 {len(per_query_docs)} ranked lists\n")
for i, (q, hits) in enumerate(zip(variations, per_query_docs), 1):
    print(f"❓ Query {i}: {q[:70]}...")
    for rank, doc in enumerate(hits):
        print(f"   rank {rank}: {doc.page_content[:60].strip()}...")
    print()

---
## 🔗 Part 6: Reciprocal Rank Fusion

The implementation is short. For every document in every list, add `1 / (rank + k)` to its running
score, then sort by the total.

One detail matters: **key documents by content, and keep the `Document` object itself.** The
original implementation used `str(doc)` as the key *and* returned that string, so downstream code
received stringified reprs rather than usable `Document` objects.

In [ ]:
# ============================================================================
# RECIPROCAL RANK FUSION: Score by rank across all result lists
# ============================================================================
def reciprocal_rank_fusion(
    results: list[list[Document]], k: int = 60
) -> list[tuple[Document, float]]:
    """Fuse ranked lists into one ranking scored by 1 / (rank + k).

    k dampens the effect of rank position: with the conventional k=60, appearing
    in many lists outweighs placing first in any single one.
    """
    fused_scores: dict[str, float] = {}
    doc_by_key: dict[str, Document] = {}

    for docs in results:
        for rank, doc in enumerate(docs):
            key = doc.page_content              # content identity, not object identity
            doc_by_key.setdefault(key, doc)     # keep the real Document
            fused_scores[key] = fused_scores.get(key, 0.0) + 1 / (rank + k)

    ranked = sorted(fused_scores.items(), key=lambda kv: kv[1], reverse=True)
    return [(doc_by_key[key], score) for key, score in ranked]


fused = reciprocal_rank_fusion(per_query_docs)

total = sum(len(h) for h in per_query_docs)
print(f"📊 {total} retrieved across {len(per_query_docs)} lists → {len(fused)} unique chunks")

---
## 🔬 Part 7: Look Inside the Fusion

This is the cell that makes RRF concrete. For each chunk it shows the fused score, **how many
queries found it**, and **at which ranks** — so you can watch consensus drive the ordering rather
than take it on faith.

In [ ]:
# ============================================================================
# INSPECTION: Fusion scores, with the votes that produced them
# ============================================================================
# Recover which queries found each chunk, and where, purely for explanation.
appearances: dict[str, list[int]] = {}
for docs in per_query_docs:
    for rank, doc in enumerate(docs):
        appearances.setdefault(doc.page_content, []).append(rank)

print(f"{'#':<3}{'score':<9}{'queries':<9}{'ranks':<14}chunk")
print("-" * 110)
for i, (doc, score) in enumerate(fused, 1):
    ranks = appearances[doc.page_content]
    preview = doc.page_content[:58].strip().replace("\n", " ")
    print(f"{i:<3}{score:<9.5f}{len(ranks):<9}{str(ranks):<14}{preview}...")

top_doc, top_score = fused[0]
print(f"\n🏆 Top chunk was found by {len(appearances[top_doc.page_content])} of "
      f"{len(per_query_docs)} queries (score {top_score:.5f})")

### 7.1 Why `k` Matters

Re-running the fusion with different `k` values shows what the constant controls. A **small** `k`
makes rank position dominate, so a single first-place hit can outrank broad agreement. A **large**
`k` flattens the curve until only the number of votes matters.

In [ ]:
# ============================================================================
# PARAMETER STUDY: How k reshapes the ranking
# ============================================================================
# Print the whole top-3 per k, not just the winner. If the ordering does not
# change on this corpus, that is a real result worth seeing rather than a
# conclusion asserted over the top of the evidence.
orderings = {}
for k in (1, 5, 60, 1000):
    ranking = reciprocal_rank_fusion(per_query_docs, k=k)
    orderings[k] = [d.page_content for d, _ in ranking]
    print(f"k={k}")
    for pos, (doc, score) in enumerate(ranking[:3], 1):
        votes = len(appearances[doc.page_content])
        preview = doc.page_content[:46].strip().replace(chr(10), " ")
        print(f"   {pos}. score={score:<9.5f} votes={votes}  {preview}...")
    print()

baseline = orderings[60]
changed = [k for k, order in orderings.items() if order != baseline]
if changed:
    print(f"📋 k values that produced a DIFFERENT ordering than k=60: {changed}")
    print("   Small k rewards placing first; large k rewards being found often.")
else:
    print("📋 Ordering was identical for every k tested.")
    print("   With only a handful of chunks and broad agreement across queries,")
    print("   consensus and rank point the same way — k only matters when they disagree.")

---
## ⚖️ Part 8: Fusion vs Flat Union

Both techniques retrieve the same chunks — the difference is **ordering**. Since RAG prompts are
context-limited, ordering decides what survives truncation, which is exactly what fusion is for.

In [ ]:
# ============================================================================
# COMPARISON: Multi-Query union ordering vs RAG-Fusion ordering
# ============================================================================
def flat_union(results: list[list[Document]]) -> list[Document]:
    """What Multi-Query does: first-seen order, rankings discarded."""
    seen, unique = set(), []
    for sublist in results:
        for doc in sublist:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                unique.append(doc)
    return unique


union_docs = flat_union(per_query_docs)
fused_docs = [doc for doc, _ in fused]

print(f"{'#':<4}{'UNION (first seen)':<56}{'FUSION (consensus first)'}")
print("-" * 112)
for i in range(max(len(union_docs), len(fused_docs))):
    u = union_docs[i].page_content[:50].strip().replace("\n", " ") if i < len(union_docs) else ""
    f = fused_docs[i].page_content[:50].strip().replace("\n", " ") if i < len(fused_docs) else ""
    print(f"{i + 1:<4}{u:<56}{f}")

same_set = {d.page_content for d in union_docs} == {d.page_content for d in fused_docs}
same_order = [d.page_content for d in union_docs] == [d.page_content for d in fused_docs]
print(f"\n📊 Same chunks retrieved? {same_set}")
print(f"📊 Same ordering?         {same_order}")
if same_set and not same_order:
    print("✅ Identical recall, different priority — fusion promoted the consensus chunks.")

---
## 🎯 Part 9: The Complete RAG Chain

The retrieval chain ends by dropping the scores, since the prompt only needs the documents — now in
fusion order, best-supported chunk first.

In [ ]:
# ============================================================================
# RETRIEVAL CHAIN: variations -> ranked lists -> RRF -> documents
# ============================================================================
retrieval_chain = (
    generate_queries
    | retriever.map()
    | reciprocal_rank_fusion
    | (lambda scored: [doc for doc, _ in scored])  # drop scores; keep fusion order
)

print("✅ RAG-Fusion retrieval chain ready")

In [ ]:
# ============================================================================
# RAG CHAIN: Fused retrieval feeds the answer prompt
# ============================================================================
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

answer = final_rag_chain.invoke({"question": question})
print(answer)

### 9.1 Verify the Context Is Real

The sibling Multi-Query notebook shipped a merge function that returned document **IDs** instead of
documents, so the prompt received opaque identifiers and the LLM answered from memory — while still
producing a fluent, plausible answer. This assertion is the cheap check that catches that class of
failure.

In [ ]:
# ============================================================================
# SANITY CHECK: Confirm retrieval actually reaches the prompt
# ============================================================================
context_docs = retrieval_chain.invoke({"question": question})

print(f"📊 Context contains {len(context_docs)} items")
print(f"📋 Type of each item: {type(context_docs[0]).__name__}")

assert all(isinstance(d, Document) for d in context_docs), "Context must hold Document objects!"
assert all(d.page_content.strip() for d in context_docs), "Documents must carry real text!"

print("✅ Context holds real Document objects with real text")
print(f"\n📄 Top-ranked chunk: {context_docs[0].page_content[:180].strip()}...")

---
## 📝 Summary

### 1. The Problem With a Flat Union
- Multi-Query discards both **rank within each list** and **agreement across lists**. A chunk every
  variation ranked first is indistinguishable from one that appeared once, near the bottom.

### 2. Reciprocal Rank Fusion
- Score each document as $\sum 1/(\text{rank} + k)$ over every list it appears in, then re-rank.
- Uses **ranks, not similarity scores** — scores from different queries are not comparable, positions are.
- **Agreement compounds**: appearing in more lists adds more terms, so consensus rises to the top.

### 3. The Role of `k`
- `k` flattens the rank curve. At the conventional `k = 60`, rank 0 and rank 3 score almost the
  same, so *how many* queries found a chunk outweighs *where* it placed.
- Part 7.1 showed a small `k` inverting that priority.

### 4. Implementation Details That Bite
- Key documents by **content**, and return the `Document` objects — not `str(doc)` reprs, and never
  bare IDs. The sibling Multi-Query notebook shipped exactly that bug, and it was invisible because
  the LLM still produced a confident answer from its own knowledge.
- Generate the variations with `with_structured_output()`. Junk lines from `split("\n")` do not
  merely add noise here — they produce ranked lists that **vote** in the fusion.

### 5. Fusion vs Union
- Same recall, different **ordering** (Part 8). Since context windows truncate, ordering decides
  what the model actually reads — which is the whole point.

### 6. Costs
- One LLM call for the variations, then N retrievals — the same as Multi-Query. RRF itself is
  essentially free: a dictionary and a sort.

### Next Steps
- Inspect these runs in LangSmith under the **RAG-Fusion** project.
- Compare with the sibling techniques: `a. Multi_Query` (same fan-out, flat merge),
  `c. Step_Back_Prompting` (generalize the question), `d. HyDE` (search with a hypothetical answer).
- RRF is not RAG-specific — it is a standard method for combining any ranked lists, such as merging
  dense and keyword (BM25) retrieval in hybrid search.